In [0]:
%sql
DROP VOLUME IF EXISTS workspace.default.checkpoints;

-- create the checkpoint volume for reading the raw tweet stream
CREATE VOLUME IF NOT EXISTS workspace.default.checkpoints;

DROP TABLE IF EXISTS workspace.default.tweets_bronze;
DROP TABLE IF EXISTS workspace.default.tweets_silver;
DROP TABLE IF EXISTS workspace.default.tweets_gold;
DROP TABLE IF EXISTS workspace.default.gold_tweet_agg;



In [0]:
# Install transformers, torch, and torchvision (required for Hugging Face models)
%pip install transformers==4.35.2 torch torchvision --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import mlflow
from mlflow import MlflowClient
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Use Unity Catalog Model Registry
# Note: Free Edition has limited permissions - model registration must be done via UI (one-time setup)
mlflow.set_registry_uri("databricks-uc")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-ab99788b-0934-40c2-808a-4408281c440a/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten
/local_disk0/.ephemeral_nfs/envs/pythonEnv-ab99788b-0934-40c2-808a-4408281c440a/lib/python3.12/site-packages/transformers/utils/__init__.py:31: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  from .generic import (
/local_disk0/.ephemeral_nfs/envs/pythonEnv-ab99788b-0934-40c2-808a-4408281c440a/lib/python3.12/site-packages/transformers/modeling_outputs.py:25: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  class BaseModelOutput(ModelOutput):
/local_disk0/.

In [0]:
# Define model details
HF_MODEL_NAME = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
UC_MODEL_NAME = "workspace.default.small_sentiment_model"

print(f"🤗 Loading Hugging Face model: {HF_MODEL_NAME}")
print(f"   This may take a few minutes on first download...")

# Load model and tokenizer from Hugging Face
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(HF_MODEL_NAME)

print(f"✅ Model loaded successfully!")
print(f"   Model size: ~67M parameters")
print(f"   Output classes: 2 (negative, positive)")

🤗 Loading Hugging Face model: distilbert/distilbert-base-uncased-finetuned-sst-2-english
   This may take a few minutes on first download...


/local_disk0/.ephemeral_nfs/envs/pythonEnv-ab99788b-0934-40c2-808a-4408281c440a/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

✅ Model loaded successfully!
   Model size: ~67M parameters
   Output classes: 2 (negative, positive)


In [0]:
# Log model to MLflow run storage (works in Free Edition)
# Note: Model REGISTRATION must be done via UI in Free Edition (see instructions below)
print(f"📦 Logging model to MLflow with transformers flavor...")

with mlflow.start_run(run_name="tweet_sentiment_hf_model") as run:
    # Log the model using transformers flavor
    model_info = mlflow.transformers.log_model(
        transformers_model={
            "model": model,
            "tokenizer": tokenizer
        },
        artifact_path="model",
        input_example=["This is a great day!"],  # Example for schema inference
        task="text-classification"
    )

    # Log model metadata for reference
    mlflow.log_param("hf_model_name", HF_MODEL_NAME)
    mlflow.log_param("task", "sentiment-classification")
    mlflow.log_param("num_labels", 2)
    mlflow.log_param("model_type", "distillbert-base")

    run_id = run.info.run_id
    model_uri = f"runs:/{run_id}/model"

    print(f"\n✅ Model logged to MLflow run: {run_id}")
    print(f"   Model URI: {model_uri}")

📦 Logging model to MLflow with transformers flavor...


2026/05/06 00:20:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-81eadd5b-7166.cloud.databricks.com/ml/experiments/4a258b4045174facaff9bed6808aa5bc/models/m-ea2d418da8304b35bc038b2855740830?o=4302636630944159


README.md: 0.00B [00:00, ?B/s]

/local_disk0/.ephemeral_nfs/envs/pythonEnv-ab99788b-0934-40c2-808a-4408281c440a/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
2026/05/06 00:20:34 INFO mlflow.transformers.signature: Running model prediction to infer the model output signature with a timeout of 180 seconds. You can specify a different timeout by setting the environment variable MLFLOW_INPUT_EXAMPLE_INFERENCE_TIMEOUT.
2026/05/06 00:21:15 WARNING mlflow.transformers.model_io: Could not specify device parameter for this pipeline type.Falling back to loading the model with the default device.


cannot access local variable 'model' where it is not associated with a value
cannot access local variable 'model' where it is not associated with a value

✅ Model logged to MLflow run: e3fb3b1d79e5415a95e497c73a74d32a
   Model URI: runs:/e3fb3b1d79e5415a95e497c73a74d32a/model


In [0]:
# Free Edition requires manual registration via Databricks UI
print(f"\n📋 ONE-TIME SETUP: Register Model via Databricks UI")
print(f"=" * 70)
print(f"\n⚠️  IMPORTANT: Only do this ONCE when first setting up the lab!")
print(f"   If model already registered, skip to verification below.\n")
print(f"📝 Manual Registration Steps:")
print(f"   1. In Databricks workspace, click 'Machine Learning' in left sidebar")
print(f"   2. Click 'Experiments' tab")
print(f"   3. Find and click the experiment containing run: {run_id}")
print(f"   4. Click the run to open run details")
print(f"   5. Scroll down to 'Artifacts' section → click 'model' folder")
print(f"   6. Click 'Register Model' button (top right)")
print(f"   7. In dialog:")
print(f"      - Model: Select 'Create New Model'")
print(f"      - Model Name: {UC_MODEL_NAME}")
print(f"      - Click 'Register'")
print(f"\n   ✅ Registration complete! Proceed to verification below.")
print(f"=" * 70)


📋 ONE-TIME SETUP: Register Model via Databricks UI

⚠️  IMPORTANT: Only do this ONCE when first setting up the lab!
   If model already registered, skip to verification below.

📝 Manual Registration Steps:
   1. In Databricks workspace, click 'Machine Learning' in left sidebar
   2. Click 'Experiments' tab
   3. Find and click the experiment containing run: e3fb3b1d79e5415a95e497c73a74d32a
   4. Click the run to open run details
   5. Scroll down to 'Artifacts' section → click 'model' folder
   6. Click 'Register Model' button (top right)
   7. In dialog:
      - Model: Select 'Create New Model'
      - Model Name: workspace.default.small_sentiment_model
      - Click 'Register'

   ✅ Registration complete! Proceed to verification below.


In [0]:
# Verify model is registered in Unity Catalog
client = MlflowClient()
try:
    model_versions = client.search_model_versions(f"name='{UC_MODEL_NAME}'")
    #for mv in model_versions:
    #    print(f"Version: {mv.version}, Status: {mv.status}, Description: {mv.description}")

    print(f"\n✅ Model registered successfully in Unity Catalog!")
    print(f"   Name: {model_versions[0].name}")
    print(f"   Description: {model_versions[0].description or 'N/A'}")

    if model_versions[0].version:
        print(f"   Latest version: {model_versions[0].version}")
        print(f"   Status: {model_versions[0].status}")
        print(f"\n   Model URI: models:/{UC_MODEL_NAME}/{model_versions[0].version}")
    else:
        print(f"   ⚠️  No versions found - complete manual registration above!")

except Exception as e:
    print(f"❌ Model not found: {e}")
    print(f"\n⚠️  Please complete the ONE-TIME manual registration above!")
    print(f"   Follow the UI registration steps, then rerun this cell to verify.")


✅ Model registered successfully in Unity Catalog!
   Name: workspace.default.small_sentiment_model
   Description: N/A
   Latest version: 1
   Status: READY

   Model URI: models:/workspace.default.small_sentiment_model/1


In [0]:
#Load the model from the URI above and execute a test inference 
pyfunc_model = mlflow.pyfunc.load_model(model_uri)

# The model is logged with an input example
input_data = pyfunc_model.input_example

# Verify the model with the provided input data using the logged dependencies.
# For more details, refer to:
# https://mlflow.org/docs/latest/models.html#validate-models-before-deployment
mlflow.models.predict(
    model_uri=model_uri,
    input_data=input_data,
    env_manager="uv",
)

2026/05/06 00:21:40 WARNING mlflow.transformers.model_io: Could not specify device parameter for this pipeline type.Falling back to loading the model with the default device.


cannot access local variable 'model' where it is not associated with a value
cannot access local variable 'model' where it is not associated with a value


2026/05/06 00:21:48 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


2026/05/06 00:21:55 INFO mlflow.utils.virtualenv: Creating a new environment in /tmp/virtualenv_envs/mlflow-6d1df9a8f6822b592db5691b50a69dff76c56ebb with python version 3.12.3 using uv
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: /tmp/virtualenv_envs/mlflow-6d1df9a8f6822b592db5691b50a69dff76c56ebb
Activate with: source /tmp/virtualenv_envs/mlflow-6d1df9a8f6822b592db5691b50a69dff76c56ebb/bin/activate
2026/05/06 00:21:56 INFO mlflow.utils.virtualenv: Installing dependencies
Using Python 3.12.3 environment at: /tmp/virtualenv_envs/mlflow-6d1df9a8f6822b592db5691b50a69dff76c56ebb
Resolved 3 packages in 140ms
 Downloaded setuptools
 Downloaded pip
Prepared 3 packages in 177ms
Installed 3 packages in 26ms
 + pip==25.0.1
 + setuptools==78.1.1
 + wheel==0.45.1
Using Python 3.12.3 environment at: /tmp/virtualenv_envs/mlflow-6d1df9a8f6822b592db5691b50a69dff76c56ebb
Resolved 129 packages in 1.21s
 Downloaded nvidia-cufile
 Downloaded kiwisolver
 Downloa

{"predictions": [{"label": "POSITIVE", "score": 0.9998801946640015}]}